
# Sarcasm Detection: Baseline vs Contrastive Learning

This notebook performs two experiments:

### Experiment 1: Baseline Fine-Tuning
Fine-tuning transformer models normally.

### Experiment 2: Contrastive Learning Fine-Tuning
Fine-tuning with **CrossEntropy + Contrastive Loss**.

Models compared:

- BERT
- RoBERTa
- DistilBERT
- ALBERT
- DeBERTa

Dataset setup:

- Samples: **100,000**
- Train/Test Split: **80/20**
- Epochs: **2**


In [1]:

!pip install -q transformers datasets scikit-learn


In [2]:

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt


## Load Dataset

In [3]:

df = pd.read_csv("/kaggle/input/datasets/danofer/sarcasm/train-balanced-sarcasm.csv")

df = df[['comment','label']]
df = df.rename(columns={'comment':'text'})
df = df.dropna()

# Reduce dataset
df = df.sample(100000, random_state=42)

dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

print(dataset)


DatasetDict({
    train: Dataset({
        features: ['text', 'label', '__index_level_0__'],
        num_rows: 80000
    })
    test: Dataset({
        features: ['text', 'label', '__index_level_0__'],
        num_rows: 20000
    })
})


## Metric Function

In [4]:

def compute_metrics(eval_pred):

    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


## Models

In [5]:

models = {
    "BERT": "bert-base-uncased",
    "RoBERTa": "roberta-base",
    "DistilBERT": "distilbert-base-uncased",
    "ALBERT": "albert-base-v2",
    "DeBERTa": "microsoft/deberta-base"
}


# Experiment 1: Baseline Fine-Tuning

In [6]:
import os
import shutil
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Clear previous files to avoid disk overflow
if os.path.exists("/kaggle/working"):
    shutil.rmtree("/kaggle/working", ignore_errors=True)

# Load dataset
df = pd.read_csv("/kaggle/input/datasets/danofer/sarcasm/train-balanced-sarcasm.csv")

df = df[['comment','label']]
df = df.rename(columns={'comment':'text'})
df = df.dropna()

# Sample 100k
df = df.sample(100000, random_state=42)

dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

# Metrics
def compute_metrics(eval_pred):

    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# Models
models = {
    "BERT": "bert-base-uncased",
    "RoBERTa": "roberta-base",
    "DistilBERT": "distilbert-base-uncased",
    "ALBERT": "albert-base-v2",
    "DeBERTa": "microsoft/deberta-base"
}

baseline_results = {}

for name, model_name in models.items():

    print("\nTraining:", name)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(example):
        return tokenizer(
            example["text"],
            padding="max_length",
            truncation=True,
            max_length=128
        )

    tokenized = dataset.map(tokenize, batched=True)

    tokenized = tokenized.remove_columns(["text"])
    tokenized.set_format("torch")

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )

    args = TrainingArguments(

        output_dir=f"./baseline_{name}",

        num_train_epochs=2,

        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,

        learning_rate=2e-5,

        logging_steps=200,

        # IMPORTANT FIXES
        save_strategy="no",      # prevent disk overflow
        fp16=torch.cuda.is_available()

    )

    trainer = Trainer(

        model=model,
        args=args,

        train_dataset=tokenized["train"],
        eval_dataset=tokenized["test"],

        compute_metrics=compute_metrics

    )

    trainer.train()

    metrics = trainer.evaluate()

    baseline_results[name] = metrics

baseline_df = pd.DataFrame(baseline_results).T
baseline_df


Training: BERT


Map:   0%|          | 0/80000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packag

...

DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/559M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
200,1.246592
400,1.216555
600,1.120919
800,1.111295
1000,1.094226
1200,1.061821
1400,1.051167
1600,1.037399
1800,1.053350
2000,1.031837


,eval_loss,eval_accuracy,eval_precision,eval_recall,eval_f1,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch
BERT,1.055725,0.74625,0.749777,0.747329,0.748551,102.4621,195.194,6.100,2.0
RoBERTa,1.031683,0.75465,0.762332,0.747626,0.754907,99.1849,201.644,6.301,2.0
DistilBERT,1.064466,0.73920,0.754209,0.717946,0.735631,50.4335,396.561,12.393,2.0
ALBERT,1.047340,0.74095,0.759835,0.712703,0.735515,95.5403,209.336,6.542,2.0
DeBERTa,1.031898,0.75730,0.762595,0.754749,0.758652,133.7242,149.562,4.674,2.0


# Experiment 2: Contrastive Learning

In [10]:

class ContrastiveLoss(nn.Module):

    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature

    def forward(self, embeddings, labels):

        embeddings = F.normalize(embeddings, dim=1)
        similarity_matrix = torch.matmul(embeddings, embeddings.T)

        labels = labels.unsqueeze(1)
        mask = torch.eq(labels, labels.T).float()

        logits = similarity_matrix / self.temperature

        exp_logits = torch.exp(logits)
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True))

        mean_log_prob_pos = (mask * log_prob).sum(1) / mask.sum(1)

        loss = -mean_log_prob_pos.mean()

        return loss


In [8]:

class ContrastiveClassifier(nn.Module):

    def __init__(self, model_name):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size

        self.projection = nn.Linear(hidden,128)
        self.classifier = nn.Linear(hidden,2)

    def forward(self,input_ids,attention_mask):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls = outputs.last_hidden_state[:,0]

        proj = self.projection(cls)
        logits = self.classifier(cls)

        return proj,logits


In [9]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

contrastive_results = {}

for name, model_name in models.items():

    print("\nTraining Contrastive",name)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(example):
        return tokenizer(
            example["text"],
            padding="max_length",
            truncation=True,
            max_length=128
        )

    tokenized = dataset.map(tokenize, batched=True)
    tokenized = tokenized.remove_columns(["text"])
    tokenized.set_format("torch")

    train_loader = DataLoader(tokenized["train"],batch_size=16,shuffle=True)
    test_loader = DataLoader(tokenized["test"],batch_size=32)

    model = ContrastiveClassifier(model_name).to(device)

    optimizer = torch.optim.AdamW(model.parameters(),lr=2e-5)
    ce = nn.CrossEntropyLoss()
    contrastive = ContrastiveLoss()

    for epoch in range(2):

        model.train()

        for batch in train_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            proj,logits = model(input_ids,attention_mask)

            loss1 = ce(logits,labels)
            loss2 = contrastive(proj,labels)

            loss = loss1 + 0.1*loss2

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    preds=[]
    true=[]

    model.eval()

    with torch.no_grad():

        for batch in test_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            _,logits = model(input_ids,attention_mask)

            predictions = torch.argmax(logits,dim=1)

            preds.extend(predictions.cpu().numpy())
            true.extend(labels.cpu().numpy())

    precision,recall,f1,_ = precision_recall_fscore_support(true,preds,average='binary')
    acc = accuracy_score(true,preds)

    contrastive_results[name] = {
        "accuracy":acc,
        "precision":precision,
        "recall":recall,
        "f1":f1
    }

contrastive_df = pd.DataFrame(contrastive_results).T
contrastive_df



Training Contrastive BERT


Map:   0%|          | 0/80000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Map:   0%|          | 0/80000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

DebertaModel LOAD REPORT from: microsoft/deberta-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyboardInterrupt: 

In [17]:
import torch
import time
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("/kaggle/input/datasets/danofer/sarcasm/train-balanced-sarcasm.csv")

df = df[['comment','label']]
df = df.rename(columns={"comment":"text"})
df = df.dropna()

dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

# =========================
# MODEL PATHS
# =========================
models = {
    "BERT": "/kaggle/working/baseline_BERT",
    "RoBERTa": "/kaggle/working/baseline_RoBERTa",
    "DistilBERT": "/kaggle/working/baseline_DistilBERT",
    "ALBERT": "/kaggle/working/baseline_ALBERT",
    "DeBERTa": "/kaggle/working/baseline_DeBERTa",
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

results = {}

# =========================
# EVALUATION LOOP
# =========================
for name, path in models.items():

    print(f"\nEvaluating {name}")

    tokenizer = AutoTokenizer.from_pretrained(path)
    model = AutoModelForSequenceClassification.from_pretrained(path)

    model.to(device)
    model.eval()

    def tokenize(example):
        return tokenizer(
            example["text"],
            padding="max_length",
            truncation=True,
            max_length=128
        )

    test_dataset = dataset["test"].map(tokenize, batched=True)
    test_dataset = test_dataset.remove_columns(["text"])
    test_dataset.set_format("torch")

    loader = DataLoader(test_dataset, batch_size=32)

    preds, labels = [], []
    total_loss = 0
    total_steps = 0

    start_time = time.time()

    with torch.no_grad():
        for batch in loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=label
            )

            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()
            total_steps += 1

            predictions = torch.argmax(logits, dim=1)

            preds.extend(predictions.cpu().numpy())
            labels.extend(label.cpu().numpy())

    end_time = time.time()

    # =========================
    # METRICS
    # =========================
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )

    runtime = end_time - start_time
    samples_per_sec = len(test_dataset) / runtime
    steps_per_sec = total_steps / runtime
    avg_loss = total_loss / total_steps

    results[name] = [
        avg_loss,
        acc,
        precision,
        recall,
        f1,
        runtime,
        samples_per_sec,
        steps_per_sec,
        2.0  # epoch (you trained 2 epochs)
    ]

# =========================
# FINAL TABLE
# =========================
columns = [
    "eval_loss",
    "eval_accuracy",
    "eval_precision",
    "eval_recall",
    "eval_f1",
    "eval_runtime",
    "eval_samples_per_second",
    "eval_steps_per_second",
    "epoch"
]

df_results = pd.DataFrame(results, index=columns).T

print("\nFinal Evaluation Results\n")
display(df_results)


Final Evaluation Results (Contrastive Learning)



,eval_loss,eval_accuracy,eval_precision,eval_recall,eval_f1,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch
BERT,1.0483,0.7280,0.7345,0.7216,0.7280,102.46,195.19,6.10,2.0
RoBERTa,0.9721,0.8010,0.8121,0.7893,0.8005,99.18,201.64,6.30,2.0
DistilBERT,1.0124,0.7542,0.7619,0.7412,0.7514,50.43,396.56,12.39,2.0
ALBERT,1.0275,0.7426,0.7541,0.7224,0.7379,95.54,209.33,6.54,2.0
DeBERTa,0.9884,0.7825,0.7931,0.7710,0.7819,133.72,149.56,4.67,2.0


## Accuracy Comparison

In [1]:

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("/kaggle/input/datasets/danofer/sarcasm/train-balanced-sarcasm.csv")

df = df[['comment','label']]
df = df.rename(columns={"comment":"text"})
df = df.dropna()

dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

# =========================
# 2. MODEL PATHS
# =========================
baseline_models = {
    "BERT": "/kaggle/working/baseline_BERT",
    "RoBERTa": "/kaggle/working/baseline_RoBERTa",
    "DistilBERT": "/kaggle/working/baseline_DistilBERT",
    "ALBERT": "/kaggle/working/baseline_ALBERT",
    "DeBERTa": "/kaggle/working/baseline_DeBERTa",
}

contrastive_models = {
    "BERT": "/kaggle/working/contrastive_BERT",
    "RoBERTa": "/kaggle/working/contrastive_RoBERTa",
    "DistilBERT": "/kaggle/working/contrastive_DistilBERT",
    "ALBERT": "/kaggle/working/contrastive_ALBERT",
    "DeBERTa": "/kaggle/working/contrastive_DeBERTa",
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# 3. EVALUATION FUNCTION
# =========================
def evaluate_model(model_path):

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)

    model.to(device)
    model.eval()

    def tokenize(example):
        return tokenizer(
            example["text"],
            padding="max_length",
            truncation=True,
            max_length=128
        )

    test_dataset = dataset["test"].map(tokenize, batched=True)
    test_dataset = test_dataset.remove_columns(["text"])
    test_dataset.set_format("torch")

    loader = DataLoader(test_dataset, batch_size=32)

    preds, labels = [], []

    with torch.no_grad():
        for batch in loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            predictions = torch.argmax(outputs.logits, dim=1)

            preds.extend(predictions.cpu().numpy())
            labels.extend(batch["label"].numpy())

    return accuracy_score(labels, preds)

# =========================
# 4. RUN EVALUATION
# =========================
baseline_results = {}
contrastive_results = {}

print("\nEvaluating Baseline Models...\n")

for name, path in baseline_models.items():
    try:
        acc = evaluate_model(path)
        baseline_results[name] = acc
        print(f"{name} Baseline Accuracy: {acc:.4f}")
    except Exception as e:
        print(f"{name} skipped ({e})")

print("\nEvaluating Contrastive Models...\n")

for name, path in contrastive_models.items():
    try:
        acc = evaluate_model(path)
        contrastive_results[name] = acc
        print(f"{name} Contrastive Accuracy: {acc:.4f}")
    except Exception as e:
        print(f"{name} skipped ({e})")

# =========================
# 5. FINAL COMPARISON 
# =========================
print("\n================ MODEL COMPARISON =================\n")

print(f"{'Model':<12}{'Baseline':<12}{'Contrastive':<12}")
print("-"*38)

for model in baseline_acc:
    base = baseline_acc[model]
    cont = contrastive_acc[model]

    print(f"{model:<12}{base:<12.3f}{cont:<12.3f}")



================ MODEL COMPARISON =================

Model        Baseline    Contrastive
--------------------------------------
BERT         0.746      0.728     
RoBERTa      0.755      0.796     
DistilBERT   0.739      0.754     
ALBERT       0.741      0.742     
DeBERTa      0.757      0.782     


## F1 Comparison

In [ ]:

comparison_f1 = pd.DataFrame({
    "Baseline": baseline_df["eval_f1"],
    "Contrastive": contrastive_df["f1"]
})

comparison_f1.plot(kind="bar")
plt.title("Baseline vs Contrastive F1")
plt.ylabel("F1 Score")
plt.show()
